# Introduction




# Prepare for analysis

## Load packages

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
#from wordcloud import WordCloud, STOPWORDS

from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

import sys
sys.path.append('..') 
from functions.visualizer import plot_count_pairs, plot_distribution_pairs
from functions.model import map_categorical_to_numeric, split_data, train_and_predict, evaluate_model

from functions.data_loader import load_data
from functions.data_inspector import (
    inspect_data,
    calculate_missing_values,
    get_frequent_data,
    get_unique_values,
)

from functions import feature_eng
from functions.name_processing import (
    parse_names,
    parse_names_column,
    aggregate_titles,
    show_title_survival_rate,
)
from functions.family_processing import create_family_type

In [ ]:
# TODO: Add all hard coded parameters like file paths or model parameters here
TRAIN_PATH = "data/train.csv"
TEST_PATH = "data/test.csv"

## Read the data

In [ ]:
train_df = load_data(TRAIN_PATH)
test_df = load_data(TEST_PATH)

# Preliminary data inspection

## Quick glimpse of the data
Use reusable inspection functions to preview both datasets.

In [ ]:
inspect_data(train_df)

In [ ]:
inspect_data(test_df)

## Few statistics on the data

### Missing data

In [ ]:
df_missing_train = calculate_missing_values(train_df)
df_missing_train

In [ ]:
df_missing_test = calculate_missing_values(test_df)
df_missing_test

### Most frequent data

In [ ]:
get_frequent_data(train_df)

In [ ]:
get_frequent_data(test_df)

### Unique values

In [ ]:
get_unique_values(train_df)

In [ ]:
get_unique_values(test_df)

# Exploratory data analysis

## Univariate analysis for all features


We show here two graphs in paralel:
* distribution of class values, split per Survived value
* comparison of class values, in train and test data


Let's first aggregate train and test data into one single dataframe, `all_df`.

In [ ]:
# TODO: turn into function

all_df = feature_eng.combine_datasets(train_df, test_df)

In [ ]:
all_df.head()

In [ ]:
color_list = ["#4C72B0", "#DD8452"]

In [ ]:
# Plot count pairs "Sex"
plot_count_pairs(all_df, col="Sex", hue="set", title="Number of passengers / Sex")

In [ ]:
# Plot distribution pairs for "Sex" and hue as "Survived"
plot_distribution_pairs(train_df, col="Sex", hue="Survived", title="Number of passengers / Sex")

In [ ]:
# Plot count pairs using all_df for the columns: Sex, Pclasss, SibSp, Parch, Embarked
for col in ['Sex', 'Pclass', 'SibSp', 'Parch', 'Embarked']:
    plot_count_pairs(all_df, col=col, hue="set", title=f"Number of passengers / {col}")

In [ ]:
# Plot count pairs using all_df for the columns: Sex, Pclasss, SibSp, Parch, Embarked and use "Survived" as hue.
for col in ['Sex', 'Pclass', 'SibSp', 'Parch', 'Embarked']:
    plot_count_pairs(train_df, col=col, hue="Survived", title=f"Passengers / {col} by Survival")

In [ ]:
# Plot distribution pairs for Age and Fare
plot_distribution_pairs(all_df, col="Age", hue="set", title="Age Distribution (Train vs Test)")
plot_distribution_pairs(all_df, col="Fare", hue="set", title="Fare Distribution (Train vs Test)")

In [ ]:
# Plot distribution pairs for Age and Fare using "Survived" as hue
plot_distribution_pairs(train_df, col="Age", hue="Survived", title="Age Distribution by Survival")
plot_distribution_pairs(train_df, col="Fare", hue="Survived", title="Fare Distribution by Survival")

## Family size


Based on SibSp (sibilings or spouse) and Parch (parents or children), we set the Family Size field.

In [ ]:
# TODO: turn into function

all_df = feature_eng.create_family_size(all_df)

In [ ]:
# TODO: turn into function
train_df = feature_eng.create_family_size(train_df)

In [ ]:
# Plot count pairs using all_df for the column "Family Size" and use "Survived" as hue.
plot_count_pairs(train_df, col="Family Size", hue="Survived", title="Family Size by Survival")

## Age interval

In [ ]:
# TODO: turn into function

all_df = feature_eng.create_age_interval(all_df)

In [ ]:
# TODO: turn into function

train_df = feature_eng.create_age_interval(train_df)

In [ ]:
all_df.head()

In [ ]:
# Plot count pairs using all_df for the column "Age Interval" and use "Survived" as hue.
plot_count_pairs(train_df, col="Age Interval", hue="Survived", title="Age Interval by Survival")

## Fare interval

In [ ]:
# TODO: turn into function

all_df = feature_eng.create_fare_interval(all_df)

In [ ]:
# TODO: turn into function

train_df = feature_eng.create_fare_interval(train_df)

In [ ]:
# Plot count pairs using all_df for the column "Fare Interval"
plot_count_pairs(all_df, col="Fare Interval", hue="set", title="Fare Interval (Train vs Test)")

Let's create a composed feature: Pclass + Sex.

In [ ]:
# TODO: turn into function

train_df = feature_eng.create_sex_pclass(train_df)


In [ ]:
# TODO: turn into function

all_df = feature_eng.create_sex_pclass(all_df)

In [ ]:
# Plot count pairs using all_df for the column "Fare Interval" and "Fare (grouped by survival)" with "Survived" as hue
plot_count_pairs(train_df, col="Fare Interval", hue="Survived", title="Fare Interval by Survival")

## Process names


When we process names, we would like to extract the following information:

- Family name - this is the first word (or few first words, if a family name with multiple names), followed by a comma  
- Title - this follows just after the comma   
- Given name - this is the word or group of words following family name  
- Maiden name - for ladies, is given between parantheses  

We start with creating a function that parses the Name string and extract (if possible) these 4 elements


In [ ]:
# Name parsing functions are now imported from functions/name_processing.py.
      
    
    

In [ ]:
all_df = parse_names_column(all_df)

In [ ]:
train_df = parse_names_column(train_df)

## Multivariate analysis


Let's look now to the interaction of multiple features.

In [ ]:
# TODO: Plot count pairs of "Age Interval" grouped by "Pclass"
plot_count_pairs(all_df, col="Age Interval", hue="Pclass", title="Age Interval grouped by Pclass")

In [ ]:
# TODO: Plot count pairs of "Age Interval" grouped by "Embarked"
plot_count_pairs(all_df, col="Age Interval", hue="Embarked", title="Age Interval grouped by Embarked")

In [ ]:
# TODO: Plot count pairs of "Pclass" grouped by "Fare Interval"
plot_count_pairs(all_df, col="Pclass", hue="Fare Interval", title="Pclass grouped by Fare Interval")

### Few more engineered data 


Let's create two more engineered features:  
* Family size interval: Single, Small, Large  
* Aggregated titles: Mr, Mrs, Master, Miss, and Rare  

In [ ]:
# TODO: turn into function
all_df = create_family_type(all_df)
train_df = create_family_type(train_df)

In [ ]:
# TODO: turn into function
train_df[["Family Size", "Family Type"]].head()

In [ ]:
all_df = aggregate_titles(all_df)
train_df = aggregate_titles(train_df)

In [ ]:
train_df[["Title", "Titles"]].head(10)

In [ ]:
show_title_survival_rate(train_df)

# Baseline model

## Feature engineering: Map categorical value to numerical values

In [ ]:
# TODO: turn into function
train_df = map_categorical_to_numeric(train_df, 'Sex')
test_df = map_categorical_to_numeric(test_df, 'Sex')

Create train-validation split.

Define predictor features and target feature.

Define the training and validation data and labels.

In [ ]:
# TODO: turn into function
predictors = ["Sex", "Pclass"]
target = 'Survived'
VALID_SIZE = 0.2

train_X, train_Y, valid_X, valid_Y = split_data(
    df=train_df, 
    predictors=predictors, 
    target=target, 
    test_size=VALID_SIZE
)

Initialize the classifiction algorithm.

In [ ]:
# TODO: could this and the following code cells be turned into a function that returns the prediction?
preds_tr, preds = train_and_predict(train_X, train_Y, valid_X)

Fit the classifier with the training data.

Predict the train data (to check the training classification error).

Predict the validation data.

# Evaluation

## Classification report for training data.

In [ ]:
evaluate_model(train_Y, preds_tr, valid_Y, preds)

Classification report for validation data.